In [221]:
# Import the libraries

%pip install yfinance 
%pip install plotly
%pip install scikit-learn
%pip install xgboost

import yfinance as yf

import pandas as pd 
import numpy as np 

import plotly.graph_objects as go 
import plotly.express as px 
#import plotly.io as pio



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### Download the data from yfinanace

In [222]:
# Download the stock data from yfinance
import yfinance as yf
# Define the stock ticker and the date range
tickers = ['GOOGL']
#tickers.append('GC=F')  # Gold futures
#tickers.append('BTC-USD')  # Bitcoin in USD
start_date = '2020-01-01'
end_date = '2026-01-01'

# Download the stock data
stocks = yf.download(tickers, start=start_date, end=end_date, auto_adjust=True)
gold = yf.download('GC=F', start=start_date, end=end_date, auto_adjust=True)
BTC = yf.download('BTC-USD', start=start_date, end=end_date, auto_adjust=True)
sp500 = yf.download('^GSPC', start=start_date, end=end_date, auto_adjust=True)
nasdaq = yf.download('^IXIC', start=start_date, end=end_date, auto_adjust=True)
dowjones = yf.download('^DJI', start=start_date, end=end_date, auto_adjust=True)   

# Arrange the data so that 'Close' price can directly be used for plotting and calculations
stocks.columns = stocks.columns.droplevel(1)  # Drop the ticker level from columns
gold.columns = gold.columns.droplevel(1)
BTC.columns = BTC.columns.droplevel(1)
sp500.columns = sp500.columns.droplevel(1)
nasdaq.columns = nasdaq.columns.droplevel(1)
dowjones.columns = dowjones.columns.droplevel(1)

# Add the major indices as features to the stocks DataFrame
stocks['sp500'] = sp500['Close']
stocks['nasdaq'] = nasdaq['Close']
stocks['dowjones'] = dowjones['Close']

# reset the index to make 'Date' a column
#stocks.reset_index(inplace=True)
# Drop the Open, High, Low columns
stocks.drop(columns=['Open', 'High', 'Low'], inplace=True)

stocks.head()

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


Price,Close,Volume,sp500,nasdaq,dowjones
Date,,,,,
2020-01-02,67.873024,27278000,3257.850098,9092.190430,28868.800781
2020-01-03,67.517967,23408000,3234.850098,9020.769531,28634.880859
2020-01-06,69.317589,46768000,3246.280029,9071.469727,28703.380859
2020-01-07,69.183701,34330000,3237.179932,9068.580078,28583.679688
2020-01-08,69.676132,35314000,3253.050049,9129.240234,28745.089844


In [223]:
# Normalize the gold and bitcoin prices to the first value to compare relative changes
def normalize_to_first(df):
    return df / df.iloc[0]

stocks['Gold'] = normalize_to_first(gold['Close'])
stocks['Bitcoin'] = normalize_to_first(BTC['Close'])


# Add the target variable, the direction of the stock price movement the next day
stocks['Target'] = np.where(stocks['Close'].shift(-1) > stocks['Close'], 1, 0)

# Add a target variable which predicts that over next 5 days, the stock price will go up at least once
stocks.head(20)

Price,Close,Volume,sp500,nasdaq,dowjones,Gold,Bitcoin,Target
Date,,,,,,,,
2020-01-02,67.873024,27278000,3257.850098,9092.190430,28868.800781,1.000000,0.970181,0
2020-01-03,67.517967,23408000,3234.850098,9020.769531,28634.880859,1.016202,1.020098,1
2020-01-06,69.317589,46768000,3246.280029,9071.469727,28703.380859,1.027353,1.079032,0
2020-01-07,69.183701,34330000,3237.179932,9068.580078,28583.679688,1.031027,1.133819,1
2020-01-08,69.676132,35314000,3253.050049,9129.240234,28745.089844,1.021581,1.122176,1
2020-01-09,70.407585,33200000,3274.699951,9203.429688,28956.900391,1.017842,1.094289,1
2020-01-10,70.862320,26258000,3265.350098,9178.860352,28823.769531,1.021646,1.134216,1
2020-01-13,71.411285,30730000,3288.129883,9273.929688,28907.050781,1.015677,1.131111,0
2020-01-14,70.943169,26076000,3283.149902,9251.330078,28939.669922,1.011742,1.226049,1


In [224]:
### Add Bollinger Bands to the stock data
# Calculate the 20-day moving average and standard deviation


def Bollinger_Bands(df, window=20, num_std_dev=2):
    df['MA20'] = df['Close'].rolling(window=window).mean()
    df['STD20'] = df['Close'].rolling(window=window).std()
    df['UpperBand'] = df['MA20'] + (df['STD20'] * num_std_dev)
    df['LowerBand'] = df['MA20'] - (df['STD20'] * num_std_dev)
    return df
stocks = Bollinger_Bands(stocks)  
#stocks.dropna(inplace=True)
#.reset_index(inplace=True)  # Drop rows with NaN values resulting from rolling calculations

stocks.head()

# Plot the stock price and Bollinger Bands
#pio.renderers.default = "browser"
def plot_bollinger_bands(stocks):
    fig = go.Figure().update_layout(width=1000, height=600, title='AAPL Stock Price with Bollinger Bands', xaxis_title='Date', yaxis_title='Price (USD)')
    fig.add_trace(go.Scatter(x=stocks.index, y=stocks['Close'], mode='lines', name='Close Price'))

    fig.add_trace(go.Scatter(x=stocks.index, 
                            y=stocks['UpperBand'], 
                            mode = 'lines', 
                            name='Upper Bollinger Band', 
                            line=dict(color='red', dash='dot')))

    fig.add_trace(go.Scatter(x=stocks.index, 
                            y=stocks['LowerBand'], 
                            mode = 'lines', 
                            name='Lower Bollinger Band', 
                            line=dict(color='red', dash='dot')))
    fig.show()


### Feature Engineering

In [225]:
# Add a feature for return %
stocks['Return'] = stocks['Close'].pct_change()

# Add momentum and rate of change features
stocks['Momentum_5'] = stocks['Close'] - stocks['Close'].shift(5)
stocks['ROC_5'] = stocks['Close'].pct_change(5)

# Add return lags
def add_return_lags(df, lag_days=5):
    for lag in range(1, lag_days + 1):
        df[f'Return_Lag{lag}'] = df['Return'].shift(lag)
    return df
stocks = add_return_lags(stocks, lag_days=4)

# Get 5 day 20 day, 50 day, and 200 day moving averages for price
stocks['MA5'] = stocks['Close'].rolling(window=5).mean()
stocks['MA50'] = stocks['Close'].rolling(window=50).mean()
stocks['MA200'] = stocks['Close'].rolling(window=200).mean()

# Add a price vs moving average feature
stocks['Price_vs_MA20'] = stocks['Close']/stocks['MA5']

# Get the lag features for the closing price
def add_price_lags(df, lag_days=2):
    for lag in range(1, lag_days + 1):
        df[f'Lag{lag}'] = df['Close'].shift(lag)
    return df
stocks = add_price_lags(stocks, lag_days=6)

# Moving average of volume for last 5 days
stocks['Volume_MA5'] = stocks['Volume'].rolling(window=5).mean()

# Moving average of volume volatility for last 5 days
stocks['Volatility'] = stocks['Close'].rolling(window=5).std()

# Capture volume spikes
stocks['VolumeSpike'] = np.where(stocks['Volume'] > 
                                 (stocks['Volume'].rolling(window=20).mean() + 
                                  2*stocks['Volume'].rolling(window=20).std()), 1, 0)

# 
def RSI(df, period=14):
    delta = df['Close'].diff()
    gains = delta.where(delta > 0, 0)
    losses = -delta.where(delta < 0, 0)
    avg_gains = gains.rolling(window=period).mean()
    avg_losses = losses.rolling(window=period).mean()
    rs = avg_gains/avg_losses
    #print(rs)
    df['RSI'] = 100 - (100/(1 + rs))
    return df

RSI(stocks, 14)

# MACD - Moving Average Convergence Divergence
def MACD(df, short_window=12, long_window=26, signal_window=9):
    df['EMA12'] = df['Close'].ewm(span=short_window).mean()
    df['EMA26'] = df['Close'].ewm(span=long_window).mean()
    df['MACD'] = df['EMA12'] - df['EMA26']
    df['SignalLine'] = df['MACD'].ewm(span=signal_window).mean()
    df['MACD_Histogram'] = df['MACD'] - df['SignalLine']
    return df

MACD(stocks)
###stocks.dropna(inplace=True)

# Add the seasonality features
stocks['DayOfWeek'] = stocks.index.dayofweek
stocks['Month'] = stocks.index.month
stocks['Quarter'] = stocks.index.quarter

stocks.head()


Price,Close,Volume,sp500,nasdaq,dowjones,Gold,Bitcoin,Target,MA20,STD20,...,VolumeSpike,RSI,EMA12,EMA26,MACD,SignalLine,MACD_Histogram,DayOfWeek,Month,Quarter
Date,,,,,,,,,,,,,,,,,,,,,
2020-01-02,67.873024,27278000,3257.850098,9092.190430,28868.800781,1.000000,0.970181,0,NaN,NaN,...,0,NaN,67.873024,67.873024,0.000000,0.000000,0.000000,3,1,1
2020-01-03,67.517967,23408000,3234.850098,9020.769531,28634.880859,1.016202,1.020098,1,NaN,NaN,...,0,NaN,67.680702,67.688668,-0.007966,-0.004426,-0.003540,4,1,1
2020-01-06,69.317589,46768000,3246.280029,9071.469727,28703.380859,1.027353,1.079032,0,NaN,NaN,...,0,NaN,68.319579,68.273923,0.045656,0.016100,0.029556,0,1,1
2020-01-07,69.183701,34330000,3237.179932,9068.580078,28583.679688,1.031027,1.133819,1,NaN,NaN,...,0,NaN,68.592348,68.528257,0.064091,0.032357,0.031734,1,1,1
2020-01-08,69.676132,35314000,3253.050049,9129.240234,28745.089844,1.021581,1.122176,1,NaN,NaN,...,0,NaN,68.886809,68.794454,0.092354,0.050205,0.042150,2,1,1


In [226]:
# Drop rows with NaN values resulting from rolling calculations
stocks.dropna(inplace=True)

def drop_columns_and_reset_index(df, columns_to_drop):
    df.drop(columns=columns_to_drop, inplace=True)
    df.reset_index(inplace=True)
    return df
# Remove the date column as it won't be used for modeling
stocks.reset_index(inplace=True)
# Remove the features that won't be used for modeling
#stocks = drop_columns_and_reset_index(stocks, ['Date', 'EMA12', 'EMA26', 'STD20', 'SignalLine'])
#stocks.drop(columns=['price'], inplace=True)

stocks.head()

Price,Date,Close,Volume,sp500,nasdaq,dowjones,Gold,Bitcoin,Target,MA20,...,VolumeSpike,RSI,EMA12,EMA26,MACD,SignalLine,MACD_Histogram,DayOfWeek,Month,Quarter
0,2020-10-15,77.135971,31210000,3483.340088,11713.870117,28494.199219,1.248409,1.596538,1,73.352692,...,0,71.664119,75.270607,74.828118,0.442489,-0.310633,0.753122,3,10,4
1,2020-10-16,77.742462,34194000,3483.810059,11671.559570,28606.310547,1.246835,1.572479,0,73.641828,...,0,70.864904,75.650892,75.043995,0.606897,-0.127127,0.734024,4,10,4
2,2020-10-19,75.870430,29734000,3426.919922,11478.879883,28195.419922,1.250508,1.630799,1,73.889307,...,0,60.958517,75.684667,75.105213,0.579455,0.014189,0.565265,0,10,4
3,2020-10-20,76.918274,41670000,3443.120117,11516.490234,28308.789062,1.253132,1.655006,1,74.115587,...,0,63.681156,75.874453,75.239514,0.634939,0.138339,0.496600,1,10,4
4,2020-10-21,78.649452,60322000,3435.560059,11484.690430,28210.820312,1.262447,1.781025,1,74.553468,...,1,65.090259,76.301376,75.492102,0.809274,0.272526,0.536748,2,10,4


# Modeling - using Logistical Regression

In [227]:
# Test train split

def test_train_split_local(df, target_col='Target', test_size=0.2):
    X = df.drop(columns=[target_col, 'Date'])
    y = df[target_col]
    from sklearn.model_selection import train_test_split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, shuffle=False)
    return X_train, X_test, y_train, y_test

X_train, X_test, y_train, y_test = test_train_split_local(stocks, target_col='Target', test_size=0.2)

print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)
from sklearn.linear_model import LogisticRegression
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)

score = lr.score(X_test, y_test)

print(f"Logistic Regression Accuracy: {score:.2f}")


(1047, 40) (262, 40) (1047,) (262,)
Logistic Regression Accuracy: 0.53


## Using other classification models like RandomForest, KNN

In [228]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)    

score = rf.score(X_test, y_test)
print(f"Random Forest Accuracy: {score:.2f}")

Random Forest Accuracy: 0.50


In [229]:
# KNN

from sklearn.neighbors import KNeighborsClassifier
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)

score = knn.score(X_test, y_test)
print(f"KNN Accuracy: {score:.2f}")

KNN Accuracy: 0.48


### Choose the Model hyperparameters using GridSearchCV

In [230]:

from sklearn.model_selection import GridSearchCV

param_grid = {
    'C': [0.01, 0.1, 1],
    'fit_intercept': [True, False]
}

grid_search = GridSearchCV(lr, param_grid)
grid_search.fit(X_train, y_train)
print(f"Best Hyperparameters: {grid_search.best_params_}")

Best Hyperparameters: {'C': 0.1, 'fit_intercept': True}


### Using models with updated parameters

In [231]:
lr_tuned = LogisticRegression(C=grid_search.best_params_['C'], 
    fit_intercept=grid_search.best_params_['fit_intercept'], max_iter=1000, random_state=42)
lr_tuned.fit(X_train, y_train)
score = lr_tuned.score(X_test, y_test)
print(f"Tuned Logistic Regression Accuracy: {score:.2f}")

Tuned Logistic Regression Accuracy: 0.53


In [232]:
stocks.info()

<class 'pandas.DataFrame'>
RangeIndex: 1309 entries, 0 to 1308
Data columns (total 42 columns):
 #   Column          Non-Null Count  Dtype        
---  ------          --------------  -----        
 0   Date            1309 non-null   datetime64[s]
 1   Close           1309 non-null   float64      
 2   Volume          1309 non-null   int64        
 3   sp500           1309 non-null   float64      
 4   nasdaq          1309 non-null   float64      
 5   dowjones        1309 non-null   float64      
 6   Gold            1309 non-null   float64      
 7   Bitcoin         1309 non-null   float64      
 8   Target          1309 non-null   int64        
 9   MA20            1309 non-null   float64      
 10  STD20           1309 non-null   float64      
 11  UpperBand       1309 non-null   float64      
 12  LowerBand       1309 non-null   float64      
 13  Return          1309 non-null   float64      
 14  Momentum_5      1309 non-null   float64      
 15  ROC_5           1309 non-null   

In [233]:
# Remove more features and see if it improves accuracy
#stocks.drop(columns=['index', 'sp500', 'VolumeSpike'], inplace=True)
stocks_cpy = stocks.copy()
stocks_cpy = drop_columns_and_reset_index(stocks_cpy, ['EMA12', 'EMA26', 'STD20', 'SignalLine', 'Return_Lag1', 'Return_Lag2'])
stocks_cpy.info()


<class 'pandas.DataFrame'>
RangeIndex: 1309 entries, 0 to 1308
Data columns (total 37 columns):
 #   Column          Non-Null Count  Dtype        
---  ------          --------------  -----        
 0   index           1309 non-null   int64        
 1   Date            1309 non-null   datetime64[s]
 2   Close           1309 non-null   float64      
 3   Volume          1309 non-null   int64        
 4   sp500           1309 non-null   float64      
 5   nasdaq          1309 non-null   float64      
 6   dowjones        1309 non-null   float64      
 7   Gold            1309 non-null   float64      
 8   Bitcoin         1309 non-null   float64      
 9   Target          1309 non-null   int64        
 10  MA20            1309 non-null   float64      
 11  UpperBand       1309 non-null   float64      
 12  LowerBand       1309 non-null   float64      
 13  Return          1309 non-null   float64      
 14  Momentum_5      1309 non-null   float64      
 15  ROC_5           1309 non-null   

In [234]:
# Using XG boost

from xgboost import XGBClassifier
xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
xgb.fit(X_train, y_train)   
score = xgb.score(X_test, y_test)
print(f"XGBoost Accuracy: {score:.2f}")

/usr/local/python/3.12.1/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [22:38:03] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBoost Accuracy: 0.48


### Check the importance of different features

In [235]:
from sklearn.ensemble import RandomForestClassifier

X_train, X_test, y_train, y_test = test_train_split_local(stocks_cpy, target_col='Target', test_size=0.2)
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

print('Random Forest Accuracy: {:.2f}'.format(rf.score(X_test, y_test)))

# print the name of the feature along with its importance
for feature, importance in zip(X_train.columns, rf.feature_importances_):
    print(f'{feature}: {importance:.4f}')

Random Forest Accuracy: 0.52
index: 0.0264
Close: 0.0293
Volume: 0.0375
sp500: 0.0296
nasdaq: 0.0278
dowjones: 0.0300
Gold: 0.0360
Bitcoin: 0.0269
MA20: 0.0237
UpperBand: 0.0264
LowerBand: 0.0253
Return: 0.0417
Momentum_5: 0.0367
ROC_5: 0.0356
Return_Lag3: 0.0379
Return_Lag4: 0.0405
MA5: 0.0240
MA50: 0.0252
MA200: 0.0280
Price_vs_MA20: 0.0372
Lag1: 0.0269
Lag2: 0.0282
Lag3: 0.0257
Lag4: 0.0225
Lag5: 0.0245
Lag6: 0.0289
Volume_MA5: 0.0444
Volatility: 0.0327
VolumeSpike: 0.0024
RSI: 0.0373
MACD: 0.0330
MACD_Histogram: 0.0342
DayOfWeek: 0.0172
Month: 0.0114
Quarter: 0.0050


In [236]:
def AddFEDFeatures(stocks, start_date, end_date):
    # Use the FED data to get the following features:
    # - Federal Funds Rate
    # - Inflation Rate
    # - Unemployment Rate
    %pip install fredapi
    from fredapi import Fred
    fred = Fred(api_key='c06a7676bffb4c8338823496ee6287bf')

    interest_rate = fred.get_series('FEDFUNDS', observation_start=start_date, observation_end=end_date)
    inflation_rate = fred.get_series('CPIAUCSL', observation_start=start_date, observation_end=end_date)
    unemployment_rate = fred.get_series('UNRATE', observation_start=start_date, observation_end=end_date)

    print(interest_rate.head())

    stocks['FederalFundsRate'] = interest_rate
    stocks['InflationRate'] = inflation_rate
    stocks['UnemploymentRate'] = unemployment_rate

